In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, gamma
from scipy.optimize import brentq
from scipy.integrate import quad

#### **Principios de cálculo de primas**
N ~ Binomial Negativa(r = 10, p = 0.3) y montos geométricos con P(Y = y) = 0.6^(y−1) · 0.4. Se calcula la distribución de S con la fórmula de Panjer, su VaR al 95% y, para una prima de 103, el parámetro del principio exponencial y del principio de riesgo ajustado.

In [2]:
# Panjer para binomial negativa
def panjer_binomial_negativa(r, p, f, s_max):
    a = 1 - p
    b = (r - 1) * (1 - p)
    g = np.zeros(s_max + 1)
    g[0] = p ** r
    for s in range(1, s_max + 1):
        j = np.arange(1, s + 1)
        g[s] = np.sum((a + b * j / s) * f[j] * g[s - j]) / (1 - a * f[0])
    return g

# Parámetros
r = 10
p = 0.3
s_max = 400

# Severidad geométrica
y = np.arange(1, s_max + 1)
f = np.zeros(s_max + 1)
f[1:] = 0.4 * 0.6 ** (y - 1)

# Distribución de S
g = panjer_binomial_negativa(r, p, f, s_max)
F_S = np.cumsum(g)
E_S = np.sum(np.arange(s_max + 1) * g)

# VaR al 95%
VaR_95 = np.argmax(F_S >= 0.95)

print(f"E[S] = {E_S:.4f}")
print(f"VaR_0.95(S) = {VaR_95}")
print(f"F_S({VaR_95 - 1}) = {F_S[VaR_95 - 1]:.6f}")
print(f"F_S({VaR_95}) = {F_S[VaR_95]:.6f}")

E[S] = 58.3333
VaR_0.95(S) = 102
F_S(101) = 0.948913
F_S(102) = 0.952023


In [ ]:
prima = 103

# Principio exponencial: p(α) = (1/α) ln M_S(α)
def M_S(alpha):
    M_Y = 0.4 * np.exp(alpha) / (1 - 0.6 * np.exp(alpha))
    return (p / (1 - (1 - p) * M_Y)) ** r

def prima_exponencial(alpha):
    return np.log(M_S(alpha)) / alpha

# M_S(α) solo existe para α < ln(1/0.88)
alpha_sol = brentq(lambda a: prima_exponencial(a) - prima, 1e-6, np.log(1 / 0.88) - 1e-9)

print("Principio exponencial")
print(f"α = {alpha_sol:.6f}")
print(f"Prima = {prima_exponencial(alpha_sol):.4f}")

In [ ]:
# Principio de riesgo ajustado: ∫ [1 − F_S(x)]^(1/ρ) dx
# Como S es discreta, 1 − F_S es constante entre enteros y la integral es una suma exacta
def prima_riesgo_ajustado(rho):
    return np.sum((1 - F_S) ** (1 / rho))

rho_sol = brentq(lambda rho: prima_riesgo_ajustado(rho) - prima, 1, 20)

print("Principio de riesgo ajustado")
print(f"ρ = {rho_sol:.6f}")
print(f"Prima = {prima_riesgo_ajustado(rho_sol):.4f}")

#### **Reaseguro de exceso de pérdida: aproximaciones para S^A**

N ~ Binomial Negativa(r = 100, p = 0.5) y montos X ~ Exp(1/3). Con reaseguro de exceso de pérdida con M = 200, la aseguradora retiene S^A = min(S, M). Se calculan E[S^A] y σ(S^A) con las aproximaciones Normal y Gamma trasladada, y la probabilidad de exceder el límite.

In [ ]:
# Parámetros
r = 100
p = 0.5
lambda_exp = 1/3
M = 200

# Momentos de X ~ Exp(1/3)
EX = 1 / lambda_exp
EX2 = 2 / lambda_exp**2
EX3 = 6 / lambda_exp**3
VarX = EX2 - EX**2
X_mu3 = EX3 - 3 * EX2 * EX + 2 * EX**3

# Momentos de N ~ BN(100, 0.5)
EN = r * (1 - p) / p
VarN = r * (1 - p) / p**2
N_mu3 = r * (1 - p) * (2 - p) / p**3

# Momentos de S
ES = EN * EX
VarS = EN * VarX + VarN * EX**2
sigma_S = np.sqrt(VarS)
S_mu3 = EN * X_mu3 + 3 * VarN * EX * VarX + N_mu3 * EX**3
gamma1_S = S_mu3 / sigma_S**3

print(f"E[S] = {ES:.4f}")
print(f"Var(S) = {VarS:.4f}")
print(f"Coeficiente de asimetría de S: {gamma1_S:.6f}")

In [ ]:
# Aproximación Normal para S^A = min(S, M)
z = (M - ES) / sigma_S
E_SA_normal = ES * norm.cdf(z) + M * (1 - norm.cdf(z)) - sigma_S * norm.pdf(z)
E_SA2_normal = (ES**2 + sigma_S**2) * norm.cdf(z) + M**2 * (1 - norm.cdf(z)) - sigma_S * (ES + M) * norm.pdf(z)
sigma_SA_normal = np.sqrt(E_SA2_normal - E_SA_normal**2)

# Aproximación Gamma trasladada para S: S ≈ a_param + G, G ~ Gamma(α, β)
alpha_S = (2 / gamma1_S)**2
beta_S = np.sqrt(alpha_S / VarS)
a_param_S = ES - alpha_S / beta_S

umbral = M - a_param_S
P_S_gt_M_gamma = gamma.sf(umbral, alpha_S, scale=1/beta_S)

E1, _ = quad(lambda u: (a_param_S + u) * gamma.pdf(u, alpha_S, scale=1/beta_S), 0, umbral)
E2, _ = quad(lambda u: (a_param_S + u)**2 * gamma.pdf(u, alpha_S, scale=1/beta_S), 0, umbral)
E_SA_gamma = E1 + M * P_S_gt_M_gamma
sigma_SA_gamma = np.sqrt(E2 + M**2 * P_S_gt_M_gamma - E_SA_gamma**2)

P_S_gt_M_normal = 1 - norm.cdf(z)

print("Aproximación Normal para S^A:")
print(f"  E[S^A] = {E_SA_normal:.4f}, σ_S^A = {sigma_SA_normal:.4f}, P(S > M) = {P_S_gt_M_normal:.4f}")
print("Aproximación Gamma trasladada para S^A:")
print(f"  E[S^A] = {E_SA_gamma:.4f}, σ_S^A = {sigma_SA_gamma:.4f}, P(S > M) = {P_S_gt_M_gamma:.4f}")

In [ ]:
s_values = np.linspace(0, ES + 3 * sigma_S, 500)
pdf_normal = norm.pdf(s_values, ES, sigma_S)
pdf_gamma = np.where(s_values > a_param_S, gamma.pdf(s_values - a_param_S, alpha_S, scale=1/beta_S), 0)

plt.figure(figsize=(10, 6))
plt.plot(s_values, pdf_normal, 'b-', linewidth=2, label='Normal')
plt.plot(s_values, pdf_gamma, 'r-', linewidth=2, label='Gamma trasladada')
plt.axvline(x=M, color='g', linestyle='--', linewidth=2, label=f'M = {M}')
plt.xlabel('S')
plt.ylabel('Densidad')
plt.title('Aproximaciones para S')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()